# Train Baseline Model

### Imports

In [1]:
import pandas as pd
import numpy as np

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, f1_score
from sklearn.metrics import brier_score_loss
from sklearn.preprocessing import StandardScaler, label_binarize

from scipy.sparse import hstack

### Load Data from Parquet File

In [2]:
df = pd.read_parquet('data.parquet')
df = df.dropna(subset=['delta'])
all_games = df['game_id']
print(f"Unique Games Loaded: {len(all_games)}")
print(f"Total Buckets Loaded: {len(df)}")

print("\nSample Reddit comments:")
comments = df['comments_list'].iloc[1450]
for i, comment in enumerate(comments[:5]):
    print(f"{comment}")

Unique Games Loaded: 50987
Total Buckets Loaded: 50987

Sample Reddit comments:
God that fox scorebug is bad. So fucking bad
This scoreboard is ass
Oh god the ugly ass Superbowl scorebug is back
I hate this score bug. Put the play clock near the game clock… makes no sense to separate them so far.
Jaysus


### Create Train/Val/Test Split
70% Train, 15% Val, 15% Test. The splitting is done by the game.

In [3]:
np.random.seed(67)

games = df['game_id'].unique().copy()
np.random.shuffle(games)

n = len(games)
train_games = games[:int(0.70 * n)]
val_games = games[int(0.70 * n):int(0.85 * n)]
test_games = games[int(0.85 * n):]

train_df = df[df['game_id'].isin(train_games)]
val_df = df[df['game_id'].isin(val_games)]
test_df = df[df['game_id'].isin(test_games)]

print(f"Train: {len(train_df)} rows ({len(train_games)} games)")
print(f"Val:   {len(val_df)} rows ({len(val_games)} games)")
print(f"Test:  {len(test_df)} rows ({len(test_games)} games)")

Train: 34876 rows (149 games)
Val:   7920 rows (32 games)
Test:  8191 rows (33 games)


### Create Windowed Data
This is so that the model can predict the next price using data from the past $N$ minutes 

In [4]:
def window_data(df, N=5):
    all_features = []
    skipped = 0

    for _, game_df in df.groupby('game_id'):
        game_df = game_df.sort_values('timestamp')
        if len(game_df) <= N:
            skipped += 1
            continue

        rows = [
            {
                # Flatten all individual comments from the N-minute window into one string
                'text_feature': " ".join(
                    comment
                    for bucket in game_df['comments_list'].iloc[i-N : i+1]
                    for comment in bucket
                ),
                'price_feature': game_df['price_close'].iloc[i-N : i+1].values,
                'count_feature': game_df['comment_count'].iloc[i-N : i+1].values,
                'score_feature': game_df['avg_score'].iloc[i-N : i+1].values,
                'label': game_df['label'].iloc[i]
            }
            for i in range(N, len(game_df))
        ]
        all_features.extend(rows)

    if skipped:
        print(f"Skipped {skipped} game(s) with fewer than {N} rows.")

    return pd.DataFrame(all_features)

### Train Model Using Different $N$ Values
Use the **val set** to select the best N, then evaluate the winner on the test set once.

In [5]:
n_values = [5, 10, 15, 20]
summary_results = []
trained_models = {}

for n in n_values:
    print(f"[INFO] Training Baseline with N={n}")

    train_win = window_data(train_df, N=n)
    val_win   = window_data(val_df,   N=n)

    vec = TfidfVectorizer(max_features=5000, sublinear_tf=True, ngram_range=(1, 2))
    X_train_text = vec.fit_transform(train_win['text_feature'])
    X_val_text   = vec.transform(val_win['text_feature'])

    cols = ['price_feature', 'count_feature', 'score_feature']
    X_train_num = np.hstack([np.vstack(train_win[c]) for c in cols])
    X_val_num   = np.hstack([np.vstack(val_win[c])   for c in cols])

    scaler = StandardScaler()
    X_train_num_s = scaler.fit_transform(X_train_num)
    X_val_num_s   = scaler.transform(X_val_num)

    X_train = hstack([X_train_text, X_train_num_s])
    X_val   = hstack([X_val_text,   X_val_num_s])

    model = LogisticRegression(class_weight='balanced', max_iter=1000)
    model.fit(X_train, train_win['label'])

    y_val_pred = model.predict(X_val)
    y_val_prob = model.predict_proba(X_val)

    acc   = accuracy_score(val_win['label'], y_val_pred)
    f1    = f1_score(val_win['label'], y_val_pred, average='macro')
    y_bin = label_binarize(val_win['label'], classes=model.classes_)
    brier = np.mean(np.sum((y_val_prob - y_bin) ** 2, axis=1))

    summary_results.append({'N': n, 'Val Accuracy': acc, 'Val Macro F1': f1, 'Val Brier': brier})
    trained_models[n] = (model, vec, scaler)

results_df = pd.DataFrame(summary_results)
print("\nVALIDATION RESULTS:")
print(results_df.to_string(index=False))

best_n = results_df.loc[results_df['Val Macro F1'].idxmax(), 'N']
print(f"\nBest N-value by Validation Macro F1: {best_n}")

print(f"\n Now, evaluting on test set, with (N={best_n})")

best_model, best_vec, best_scaler = trained_models[best_n]
test_win = window_data(test_df, N=best_n)

X_test_text = best_vec.transform(test_win['text_feature'])
X_test_num = np.hstack([np.vstack(test_win[c]) for c in cols])
X_test_num_s = best_scaler.transform(X_test_num)
X_test = hstack([X_test_text, X_test_num_s])

y_test_pred = best_model.predict(X_test)
y_test_prob = best_model.predict_proba(X_test)

print(classification_report(test_win['label'], y_test_pred, digits=3))

y_test_bin = label_binarize(test_win['label'], classes=best_model.classes_)
brier_test = np.mean(np.sum((y_test_prob - y_test_bin) ** 2, axis=1))
print(f"Brier Score: {brier_test:.4f}  (note that random guessing=0.667, simply guessing the majority class = 0.49)")

[INFO] Training Baseline with N=5
[INFO] Training Baseline with N=10
[INFO] Training Baseline with N=15
Skipped 1 game(s) with fewer than 15 rows.
[INFO] Training Baseline with N=20
Skipped 1 game(s) with fewer than 20 rows.

VALIDATION RESULTS:
 N  Val Accuracy  Val Macro F1  Val Brier
 5      0.476546      0.449045   0.618885
10      0.470658      0.448424   0.618696
15      0.464919      0.447300   0.619725
20      0.460852      0.444632   0.620264

Best N-value by Validation Macro F1: 5

 Now, evaluting on test set, with (N=5)
              precision    recall  f1-score   support

        DOWN      0.335     0.349     0.342      2204
        FLAT      0.680     0.597     0.636      3702
          UP      0.331     0.387     0.357      2120

    accuracy                          0.473      8026
   macro avg      0.448     0.444     0.445      8026
weighted avg      0.493     0.473     0.481      8026

Brier Score: 0.6150  (note that random guessing=0.667, simply guessing the majorit